# CUDA/PyTorch Reproduction for Perishable Inventory Value Iteration

This notebook reproduces the value-iteration and GPU timing experiments from the paper.
The main focus is the two-product substitution model and the Table 4 timing experiment.

**Main reproduction setting used for the final result**

- GPU implementation: batched PyTorch tensor computation
- Final dtype: `torch.float64`
- Final batch size: `256`
- Number of value-iteration iterations: `100`
- Hardware used in the experiments: Tesla T4 GPU

The notebook is organized as follows:

1. One-product baseline and simulation checks.
2. Two-product substitution model and exact event aggregation.
3. CPU and single-state PyTorch reference implementations.
4. Non-batched optimized PyTorch VI.
5. Batched PyTorch VI used for the final Table 4 reproduction.
6. Final P1–P4 timing comparison with the paper.


In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import exp, factorial,comb
from itertools import product
from functools import lru_cache
import time
from math import ceil

## 1. One-product baseline

The paper first introduces a single-product perishable inventory model.
I reproduce this simpler setting first because it verifies the FIFO inventory transition,
the censored-demand treatment, and the average-profit value-iteration update before moving to the two-product substitution model.

This section is mainly a warm-up and sanity check; the final Table 4 reproduction uses the two-product model below.


In [2]:
## simple case of one product VI(Algorithm 1)
#parameters
s = 1
c = 0.5
mu = 5
M = 2
eps = 1e-4
Q_max = 9

#main demo
states = list(product(range(Q_max + 1), repeat=M))
state_to_idx = {state: i for i,state in enumerate(states)}
N = len(states)

def poisson_pdf(n, mu):
    return exp(-mu) * (mu**n) / factorial(n)

def poisson_tail(n, mu):
    if n <= 0.0:
        return 1.0
    return 1 - sum(poisson_pdf(i, mu) for i in range(n))

def transition(state, q, d):
    '''
    state = (I1,I2,...IM)
    q = today's order quantity, arrives tomorrow
    d = today's demand
    F(Q,I,d) in the paper
    '''
    inv = list(state)
    remain_demand = d

    for m in range(M):
        sold = min(inv[m], remain_demand)
        remain_demand -= sold
        inv[m] -= sold

    next_state = inv[1:] + [q]
    return tuple(next_state)


def Esale(state):
    Y = sum(state)
    if Y == 0.0:
        return 0.0
    esale = 0.0
    for d in range(Y):
        esale += poisson_pdf(d,mu) * d
    esale += Y * poisson_tail(Y, mu)
    return esale

expected_sales = np.array([Esale(state) for state in states])

def expected_future_value(state, q, W):
    '''
    Compute E[W(F(Q,I,d))]
    '''
    Y = sum(state)
    v = 0.0

    for d in range(Y):
        p = poisson_pdf(d, mu)
        next_state = transition(state, q, d)
        k = state_to_idx[next_state]
        v += p * W[k]

    tail_prob = poisson_tail(Y, mu)
    tail_state = transition(state, q, Y)
    k_tail = state_to_idx[tail_state]
    v += tail_prob * W[k_tail]
    return v

def value_iteration():
    '''
    V = track value function
    policy = keep track of q*
    iteration = number of interation
    estimated_pi = estimated long term average profit
    '''
    V = expected_sales.copy()
    iteration = 0
    policy = np.zeros(N, dtype=int)
    while True:
        iteration += 1
        W = V.copy()

        for j, state in enumerate(states):
            best_v = -np.inf
            best_q = 0
            for q in range(Q_max + 1):
                future = expected_future_value(state, q, W)
                value_q = s * expected_sales[j] + future - c * q
                if value_q >= best_v:
                    best_v = value_q
                    best_q = q
            V[j] = best_v
            policy[j] = best_q

        diff = V - W
        span = diff.max() - diff.min()

        print(f"Iteration {iteration:2d}: span = {span:.10f}")

        if span < eps:
            estimated_pi = 0.5 * (diff.max() + diff.min())
            break

    return V, policy, iteration, estimated_pi


### 1.1 Reproducing the one-product Section 2.3 result

This cell runs value iteration under the one-product setting and checks whether the estimated average profit,
convergence iteration count, maximum order quantity, and waste percentage are consistent with the paper.
It is useful for validating the basic transition and tail-probability handling.


In [ ]:
# reproduce result in Section2.3
V, policy, iteration, pi_est= value_iteration()

print("\nConverged.")
print("Iterations:", iteration)
print("Estimated average profit pi:", pi_est)

### 1.2 Simulation check under the VI policy

After obtaining a VI policy, I simulate the inventory process to verify that the simulated long-run profit
agrees with the value-iteration estimate. This is a Monte Carlo check rather than part of the GPU timing experiment.


In [ ]:
def simulate_vi_policy(policy, T=400_000, seed=0, burn_in=1000):
    """
    Simulate the inventory system using the VI-derived policy.
    """

    rng = np.random.default_rng(seed)

    state = tuple([0] * M)

    total_profit = 0.0
    total_waste = 0.0
    total_order = 0.0

    for t in range(T + burn_in):
        j = state_to_idx[state]
        q = int(policy[j])

        d = rng.poisson(mu)

        Y = sum(state)
        sale = min(d, Y)

        waste = max(state[0] - d, 0)
        profit = s * sale - c * q
        next_state = transition(state, q, d)

        if t >= burn_in:
            total_profit += profit
            total_waste += waste
            total_order += q

        state = next_state

    avg_profit = total_profit / T
    waste_percentage = total_waste / total_order if total_order > 0 else 0.0

    return avg_profit, waste_percentage

# Source: Use ChatGPT to write testing code
def check_against_paper(policy, pi_est, iteration):
    """
    Compare your reproduction results with the paper's Section 2.3 benchmark.
    """

    # Paper benchmark for one-product M = 2 case
    paper_pi = 2.215
    paper_waste = 0.0578
    paper_iterations = 12
    paper_policy_max_q = 7

    sim_profit, sim_waste = simulate_vi_policy(policy=policy, T=400_000, seed=0, burn_in=1000)
    max_q_in_policy = int(np.max(policy))

    print("\n================ Reproduction Check ================")
    print("Parameter setting:")
    print(f"s={s}, c={c}, mu={mu}, M={M}, Q_max={Q_max}, epsilon={eps}")
    print()

    print("1. VI convergence iterations")
    print(f"Your result:   {iteration}")
    print(f"Paper result:  about {paper_iterations}")
    print(f"Difference:    {iteration - paper_iterations}")
    print()

    print("2. Estimated pi from value iteration")
    print(f"Your pi_est:   {pi_est:.6f}")
    print(f"Paper pi:      {paper_pi:.6f}")
    print(f"Abs diff:      {abs(pi_est - paper_pi):.6f}")
    print()

    print("3. Maximum order quantity used by optimal policy")
    print(f"Your max q*:   {max_q_in_policy}")
    print(f"Paper says:    never orders more than {paper_policy_max_q}")
    print()

    print("4. Simulation under VI policy")
    print(f"Your sim profit:  {sim_profit:.6f}")
    print(f"Paper pi/profit:  {paper_pi:.6f}")
    print(f"Abs diff:         {abs(sim_profit - paper_pi):.6f}")
    print()

    print("5. Waste percentage under VI policy")
    print(f"Your waste:    {100 * sim_waste:.3f}%")
    print(f"Paper waste:   {100 * paper_waste:.3f}%")
    print(f"Abs diff:      {100 * abs(sim_waste - paper_waste):.3f} percentage points")
    print()

    # Loose tolerance checks
    pass_pi = abs(pi_est - paper_pi) < 0.03
    pass_sim_profit = abs(sim_profit - paper_pi) < 0.05
    pass_waste = abs(sim_waste - paper_waste) < 0.015
    pass_max_q = max_q_in_policy <= paper_policy_max_q

    print("Pass / warning summary:")
    print(f"pi_est close to paper:        {pass_pi}")
    print(f"simulation profit close:      {pass_sim_profit}")
    print(f"waste close to paper:         {pass_waste}")
    print(f"max q consistent with paper:  {pass_max_q}")

    if pass_pi and pass_sim_profit and pass_waste and pass_max_q:
        print("\nOverall: reproduction looks consistent with the paper.")
    else:
        print("\nOverall: some results differ. Check transition function, tail probability, or simulation waste definition.")


check_against_paper(policy, pi_est, iteration)

### 1.3 Base-stock policy baseline

The base-stock policy is included as a benchmark policy. It helps compare the optimized VI policy against a simpler heuristic.
This section is not needed for Table 4, but it provides context for the inventory-control setting.


In [3]:
# part2 base-stock policy
def basestock(state, S, mu, Q_max = None):
    total_inv = sum(state)
    estimated_waste  = max(0, state[0] - mu)
    q = max(0, S- total_inv + estimated_waste)
    q = int(round(q))
    if Q_max is not None:
        q = min(q, Q_max)
    return q

## 2. Two-product substitution model

The paper then extends the model to two substitutable products.
Product `b` can be substituted by product `a`, so demand for `b` that cannot be satisfied may create additional effective demand for `a`.

In this notebook, a state is represented as:


$((I^a_1, \ldots, I^a_M), (I^b_1, \ldots, I^b_M))$.


The transition function applies FIFO sales for each product and appends the new order quantity as the newest inventory batch.


In [ ]:
#parameters
ca = 0.5
cb = 0.5
sa = 1
sb = 1
Qa = 3
Qb = 3
M = 2
mu_a = 5
mu_b = 5
gamma = 0.5

In [ ]:
## part3 VI for 2 products(Algorithm 2)
#main demo
def binom_pdf(u,x,gamma):
    if u < 0.0 or u > x:
        return 0.0
    return comb(x, u) *(gamma ** u) * ((1 - gamma) ** (x - u))

states_a = list(product(range(Qa + 1), repeat=M))
states_b = list(product(range(Qb + 1), repeat=M))
states_2 = list(product(states_a, states_b))
state_to_index = {state: i for i, state in enumerate(states_2)}
N2 = len(states_2)

def transition_two_products(state, qa, qb, da, db, u):
    '''
    b can be substituted by a, u = substitution amount
    '''
    state_a = state[0]
    state_b = state[1]
    next_b = transition(state_b, qb, db)
    next_a = transition(state_a, qa, da + u)
    next_state = tuple([next_a, next_b])
    return next_state

def compute_rewards(state, da, db, u):
    state_a = state[0]
    state_b = state[1]
    Y_a = sum(state_a)
    Y_b = sum(state_b)
    sale_b = min(db, Y_b)
    sale_a = min(da + u, Y_a)
    reward = sa * sale_a + sb * sale_b
    return sale_a, sale_b, reward

def compute_waste(state, da, db, u):
    state_a = state[0]
    state_b = state[1]
    waste_a = max(0, state_a[0] - da - u)
    waste_b = max(0, state_b[0] - db)
    return waste_a, waste_b

@lru_cache(None)
def compute_pu(u, yb, tol = 1e-14):
    '''
    compute the distribution of substitution demand u
    '''
    tot = 0.0
    x = max(1,u)
    while True:
        db = yb + x
        tot += poisson_pdf(db,mu_b) * binom_pdf(u, x, gamma)
        remain_tail = poisson_tail(db + 1, mu_b)
        if remain_tail < tol:
            break
        x += 1
    return tot

@lru_cache(None)
def compute_pa(ya, yb):
    '''
    compute the probablity of a when b has an inventory of yb
    b_tail = P(Db>yb), z_tail = p(Db>yb, z>ya)
    '''
    b_tail = poisson_tail(yb + 1,mu_b)
    if ya == 0.0:
        return tuple(), b_tail
    pz = []
    for z in range(ya):
        prob_z = 0.0
        for u in range(z+1):
            da = z - u
            prob_z += poisson_pdf(da, mu_a) * compute_pu(u, yb)
        pz.append(prob_z)

    z_tail = b_tail - sum(pz)
    z_tail = max(0, z_tail)
    return tuple(pz), z_tail

def compute_action_value(state, qa, qb, W):
    state_a = state[0]
    state_b = state[1]
    Ya = sum(state_a)
    Yb = sum(state_b)
    total = 0.0

    #scenario1: db <= Yb
    for db in range(Yb + 1):
        pb = poisson_pdf(db, mu_b)
        sale_b = db
        next_b = transition(state_b, qb, db)
        # da < Ya
        for da in range(Ya):
            pa = poisson_pdf(da, mu_a)
            sale_a = da
            next_a = transition(state_a, qa, da)
            next_state = (next_a, next_b)
            k = state_to_index[next_state]
            reward = sa * sale_a + sb * sale_b
            total += pa * pb * (reward + W[k])

        # da >= Ya
        pa_tail = poisson_tail(Ya, mu_a)
        next_a_tail = transition(state_a, qa, Ya)
        next_state_tail = (next_a_tail, next_b)
        k_tail = state_to_index[next_state_tail]
        total += pb * pa_tail * (sa * Ya + sb * sale_b + W[k_tail])

    #scenario2 : db > Yb

    next_b_out = transition(state_b, qb, Yb)
    pz, z_tail = compute_pa(Ya, Yb)

    # da < Ya
    for z,pz in enumerate(pz):
        next_a = transition(state_a, qa, z)
        next_state_out = (next_a, next_b_out)
        k1 = state_to_index[next_state_out]
        total += pz * (sa * z + sb * Yb + W[k1])

    # da >= Ya
    next_a_out_tail = transition(state_a, qa, Ya)
    next_state_both_out = (next_a_out_tail, next_b_out)
    k2 = state_to_index[next_state_both_out]
    total += z_tail * (sa * Ya + sb * Yb + W[k2])

    total -= ca * qa + cb * qb
    return total

def two_products_VI(max_iter = 100):
    V = np.zeros(N2)
    policy = np.zeros((N2,2) ,dtype=int)
    for iter in range(1, max_iter + 1):
        W = V.copy()
        for j, state in enumerate(states_2):
            best_value = -np.inf
            best_qa = 0
            best_qb = 0
            for qa in range(Qa + 1):
                for qb in range(Qb + 1):
                    value = compute_action_value(state, qa, qb, W)

                    if value > best_value:
                        best_value = value
                        best_qa = qa
                        best_qb = qb
            V[j] = best_value
            policy[j,0] = best_qa
            policy[j,1] = best_qb
        diff = V - W
        span = diff.max() - diff.min()
        est_pi = 0.5 * (diff.max() + diff.min())
        if span < eps:
            print("Converged")
            return V, policy, est_pi, iter

    print("Reached max iteration")
    return V, policy, est_pi, max_iter


### 2.1 Two-product base-stock simulation

This cell implements a two-product base-stock simulation as an additional benchmark.
It is not the final GPU reproduction target, but it provides a useful comparison with the optimized VI policy.


In [ ]:
# Section 3.2
def simulate_2_base_stock(Sa, Sb, T = 400000, seed = 0, ):
    rng = np.random.default_rng(seed)
    state = ((0,)*M, (0,)*M)
    total_profit = 0.0
    total_order_a = 0.0
    total_order_b = 0.0
    total_sales_a = 0.0
    total_sales_b = 0.0
    total_waste_a = 0.0
    total_waste_b = 0.0

    for t in range(T):
        state_a = state[0]
        state_b = state[1]
        da = rng.poisson(mu_a)
        db = rng.poisson(mu_b)
        qa = basestock(state_a, Sa, mu_a)
        qb = basestock(state_b, Sb, mu_b)
        Ya = sum(state_a)
        Yb = sum(state_b)

        short_b = max(db - Yb, 0)
        u = rng.binomial(short_b, gamma)
        sale_a = min(da + u, Ya)
        sale_b = min(db, Yb)
        waste_a = max(state_a[0] - da - u, 0)
        waste_b = max(state_b[0] - db, 0)
        profit = sa * sale_a + sb * sale_b - ca * qa - cb * qb
        next_state = transition_two_products(state, qa, qb, da, db, u)

        total_profit += profit
        total_order_a += qa
        total_order_b += qb
        total_sales_a += sale_a
        total_sales_b += sale_b
        total_waste_a += waste_a
        total_waste_b += waste_b
        state = next_state

    avg_profit = total_profit / T
    avg_order_a = total_order_a / T
    avg_order_b = total_order_b / T
    avg_sales_a = total_sales_a / T
    avg_sales_b = total_sales_b / T
    avg_waste_units_a = total_waste_a / T
    avg_waste_units_b = total_waste_b / T
    waste_perc_a = total_waste_a / total_order_a if total_order_a > 0 else 0.0
    waste_perc_b = total_waste_b / total_order_b if total_order_b > 0 else 0.0

    return {
        "avg_profit": avg_profit,
        "waste_perc_a": waste_perc_a,
        "waste_perc_b": waste_perc_b,
        "avg_order_a": avg_order_a,
        "avg_order_b": avg_order_b,
        "avg_sales_a": avg_sales_a,
        "avg_sales_b": avg_sales_b,
        "avg_waste_units_a": avg_waste_units_a,
        "avg_waste_units_b": avg_waste_units_b,
        "flow_balance_a": avg_order_a - avg_sales_a - avg_waste_units_a,
        "flow_balance_b": avg_order_b - avg_sales_b - avg_waste_units_b,
    }


## 3. Exact event aggregation for demand and substitution

Directly summing over all raw demand realizations is expensive because Poisson demand has infinite support.
The paper's idea is to only distinguish events that lead to different observed sales and next states.
Tail events that produce the same effective transition are merged.

Here `build_event_for_state(state)` constructs aggregated events:

- `da_eff`: effective demand for product `a`;
- `db_eff`: effective demand for product `b`;
- `prob`: probability mass of that aggregated event.

This is the key step that makes the no-truncation implementation feasible.


In [7]:
@lru_cache(None)
def build_event_for_state(state):
    state_a = state[0]
    state_b = state[1]
    Ya = sum(state_a)
    Yb = sum(state_b)

    da_list = []
    db_list = []
    prob_list = []

    #Scenario1: db <= Yb:
    for db in range(Yb + 1):
        prob_b = poisson_pdf(db, mu_b)

        #case1: da < Ya:
        for da in range(Ya):
            prob_a = poisson_pdf(da, mu_a)
            da_list.append(da)
            db_list.append(db)
            prob_list.append(prob_a * prob_b)

        #case 2: da >= Ya:
        pa_tail = poisson_tail(Ya, mu_a)
        da_list.append(Ya)
        db_list.append(db)
        prob_list.append(pa_tail * prob_b)

    #Scenario2: db > Yb:

    pz_list, pz_tail = compute_pa(Ya, Yb)

    for z,pz in enumerate(pz_list):
        da_list.append(z)
        db_list.append(Yb)
        prob_list.append(pz)

    da_list.append(Ya)
    db_list.append(Yb)
    prob_list.append(pz_tail)

    return (
        np.array(da_list, dtype = np.int64),
        np.array(db_list, dtype = np.int64),
        np.array(prob_list, dtype = np.float64)
    )

## 4. CPU reference implementation for one-state action values

This CPU function computes all action values \(Fv(q_a, q_b)\) for a fixed state.
It is intentionally simple and is used as a correctness reference for the PyTorch implementation.

The final timing results do **not** use this CPU function, because pure Python loops are too slow for P1–P4.


In [8]:
def compute_all_values_for_state_cpu(state, W):
    state_a = state[0]
    state_b = state[1]
    Ya = sum(state_a)
    Yb = sum(state_b)

    da_list, db_list, prob_list = build_event_for_state(state)
    Fv = np.zeros((Qa + 1, Qb + 1), dtype=np.float64)

    for qa in range(Qa + 1):
        for qb in range(Qb + 1):
            total = 0.0

            for da, db, p in zip(da_list, db_list, prob_list):
                da = int(da)
                db = int(db)

                sale_a = min(da, Ya)
                sale_b = min(db, Yb)

                next_state = transition_two_products(state, qa, qb, da, db, 0)
                k = state_to_index[next_state]
                total += p * (sa * sale_a + sb * sale_b + W[k])
            total -= ca * qa + cb * qb
            Fv[qa, qb] = total

    best_qa, best_qb = np.unravel_index(np.argmax(Fv), Fv.shape)
    best_value = Fv[best_qa, best_qb]

    return Fv, best_value, best_qa, best_qb

### 4.1 Inventory encoding and PyTorch transition utilities

To evaluate future values on GPU, each next inventory state must be mapped to an integer index.
The encoding treats each product's inventory vector as a base-\((Q+1)\) number.
For two products, the combined state index is:


$\text{idx} = \text{idx}_a \cdot (Q_b+1)^M + \text{idx}_b$.



In [9]:
def encode_inventory_torch(inv, Q, M):
    '''
    map index number to inventory states for each product, return value is a tensor
    '''
    idx = torch.zeros_like(inv[0], dtype=torch.long)
    for r in range(M):
        power = M - 1 - r
        idx = idx + inv[r].long() * ((Q+1)**power)

    return idx

def next_inventory_torch(state_tuple, qmat, dmat, Q, M):
    '''
    for each product, generate next state tensor; matches cpu version of transition
    state_tuple = (I1,I2,...IM)
    qmat = torch tensor, order quantity broadcasted over event
    dmat = torch tensor
    '''
    device = dmat.device
    inv = torch.tensor(state_tuple, device=device, dtype=torch.long)
    next_state = []
    pre_sum = torch.tensor(0, device=device, dtype=torch.long)
    for r in range(M-1):
        pre_sum = pre_sum + inv[r]
        unmet = torch.clamp(dmat - pre_sum, min = 0)
        next_r = torch.clamp(inv[r+1] - unmet, min = 0)
        next_r = torch.clamp(next_r, min = 0, max = Q)
        next_state.append(next_r.long())
    next_state.append(qmat.long())
    return next_state

### 4.2 Single-state PyTorch action-value computation

This function computes the full \(Fv\) matrix for one state using PyTorch tensors.
It is mainly a debugging and sanity-check version because it returns the whole matrix to CPU.

The final GPU timing experiment uses the batched implementation later in the notebook.


In [10]:
def torch_compute_all_values(state, W, device = None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    state_a = state[0]
    state_b = state[1]
    Ya = sum(state_a)
    Yb = sum(state_b)

    W_torch = torch.as_tensor(W, device=device, dtype=torch.float64)
    da_list, db_list, prob_list = build_event_for_state(state)
    da_list_torch = torch.as_tensor(da_list, device=device, dtype=torch.long)
    db_list_torch = torch.as_tensor(db_list, device=device, dtype=torch.long)
    prob_list_torch = torch.as_tensor(prob_list, device=device, dtype=torch.float64)

    qa_arr = torch.arange(Qa + 1, device=device, dtype=torch.long)
    qb_arr = torch.arange(Qb + 1, device=device, dtype=torch.long)
    qa_grid, qb_grid = torch.meshgrid(qa_arr, qb_arr, indexing='ij')

    qa_flat = qa_grid.reshape(-1)
    qb_flat = qb_grid.reshape(-1)

    A = qa_flat.shape[0]
    E = da_list_torch.shape[0]

    qa_mat = qa_flat[:, None].expand(A, E)
    qb_mat = qb_flat[:, None].expand(A, E)

    da_mat = da_list_torch[None, :].expand(A, E)
    db_mat = db_list_torch[None, :].expand(A, E)
    prob_mat = prob_list_torch[None, :].expand(A, E)

    next_state_a = next_inventory_torch(state_a, qa_mat, da_mat, Qa, M)
    next_state_b = next_inventory_torch(state_b, qb_mat, db_mat, Qb, M)

    index_a = encode_inventory_torch(next_state_a, Qa, M)
    index_b = encode_inventory_torch(next_state_b, Qb, M)
    next_index = index_a * (Qb + 1)**M + index_b
    next_index = next_index.long()

    sale_a = torch.minimum(da_mat, torch.tensor(Ya, device=device, dtype=torch.long))
    sale_b = torch.minimum(db_mat, torch.tensor(Yb, device=device, dtype=torch.long))

    reward = sa * sale_a.to(torch.float64) + sb * sale_b.to(torch.float64)
    values = (prob_mat * (reward + W_torch[next_index])).sum(dim = 1)
    order_cost = ca  * qa_flat.to(torch.float64) + cb * qb_flat.to(torch.float64)
    values -= order_cost
    Fv = values.reshape(Qa + 1, Qb + 1)

    best_idx = torch.argmax(Fv.reshape(-1))
    best_qa = int(best_idx // (Qb + 1))
    best_qb = int(best_idx % (Qb + 1))
    best_value = float(Fv.reshape(-1)[best_idx].item())

    return Fv.detach().cpu().numpy(), best_value, best_qa, best_qb

## Appendix-style reference: Algorithm 3 non-batched VI

This cell implements an Algorithm 3-style value-iteration loop:
it iterates over states and computes the best action for each state.

It is useful for understanding the sequential VI structure, but it is not the final Table 4 implementation.
The final reproduction uses the batched PyTorch VI, which is much faster.


In [ ]:
## reproduce of Algorithm3 but with inefficient computation 
def torch_two_products_VI(max_iter = 100):
    V = np.zeros(N2, dtype=np.float64)
    policy  = np.zeros((N2, 2), dtype=int)
    pi_est = None

    for iteration in range(max_iter):
        W = V.copy()
        for j,state in enumerate(states_2):
            if j % 500 == 0:
                print(f'Iteration {iteration}, state{j}/{N2}')

            Fv, best_value, best_qa, best_qb = (torch_compute_all_values(state, W))

            V[j] = best_value
            policy[j,0] = best_qa
            policy[j,1] = best_qb

        diff = V - W
        span = diff.max() - diff.min()
        pi_est = 0.5 * (diff.max() + diff.min())

        print(
            f"Iteration {iteration:2d}: "
            f"span={span:.8f}, pi_est={pi_est:.6f}"
        )

        if span < eps:
            print("Converged")
            return V, policy, pi_est, iteration

    print("Reached max iteration")
    return V, policy, pi_est, max_iter

## 5. Timing utilities for Table 4 reproduction

Table 4 in the paper compares sequential and GPU runtimes for four problem instances P1–P4.
The timing functions below synchronize CUDA before and after timed regions to avoid undercounting asynchronous GPU work.


In [ ]:
## speed check (reproduce table 4 in the paper)
from math import ceil

def sync_if_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def cpu_vs_torch_one_state(state, W, repeat=50, warmup=5, verbose=True):
    '''
    Compare CPU mock and Torch mock for one fixed state.
    CPU:
        compute_all_values_for_state_cpu(state, W)
    Torch:
        torch_compute_all_values(state, W)
    '''

    # correctness check
    Fv_cpu, best_cpu, qa_cpu, qb_cpu = compute_all_values_for_state_cpu(state, W)
    Fv_torch, best_torch, qa_torch, qb_torch = torch_compute_all_values(state, W)

    max_diff = np.max(np.abs(Fv_cpu - Fv_torch))
    mean_diff = np.mean(np.abs(Fv_cpu - Fv_torch))

    # warmup
    for _ in range(warmup):
        compute_all_values_for_state_cpu(state, W)

    start = time.perf_counter()
    for _ in range(repeat):
        compute_all_values_for_state_cpu(state, W)
    cpu_time = (time.perf_counter() - start) / repeat

    for _ in range(warmup):
        torch_compute_all_values(state, W)
    sync_if_cuda()

    start = time.perf_counter()
    for _ in range(repeat):
        torch_compute_all_values(state, W)
    sync_if_cuda()
    torch_time = (time.perf_counter() - start) / repeat

    speedup = cpu_time / torch_time if torch_time > 0 else float("inf")

    da_eff, db_eff, prob = build_event_for_state(state)
    nblocks = (Qa + 1) * (Qb + 1)
    nthreads_equiv = 32 * ceil(len(prob) / 32)

    if verbose:
        print("State:", state)
        print("CPU best:", best_cpu, qa_cpu, qb_cpu)
        print("Torch best:", best_torch, qa_torch, qb_torch)
        print("Max abs diff:", max_diff)
        print("Mean abs diff:", mean_diff)
        print()
        print("Algorithm 4-like config:")
        print("nblocks =", nblocks)
        print("aggregated events E =", len(prob))
        print("nthreads equivalent rounded =", nthreads_equiv)
        print("probability mass =", prob.sum())
        print()
        print("Timing:")
        print(f"CPU avg time:   {cpu_time:.8f} sec")
        print(f"Torch avg time: {torch_time:.8f} sec")
        print(f"Speedup:        {speedup:.3f}x")

    return {
        "state": state,
        "cpu_time": cpu_time,
        "torch_time": torch_time,
        "speedup": speedup,
        "max_diff": max_diff,
        "mean_diff": mean_diff,
        "E": len(prob),
        "nblocks": nblocks,
        "nthreads_equiv": nthreads_equiv,
    }

### 5.1 Precomputing action tensors

In every state, the possible action pairs are:


$q_a = 0,\ldots,Q_a, \qquad q_b = 0,\ldots,Q_b$.


These action grids do not depend on the state or iteration, so they are cached once per P1–P4 configuration.
This avoids repeatedly rebuilding the same `qa_flat`, `qb_flat`, and order-cost tensors.


In [ ]:
ACTION_CACHE = {}

def prepare_action_tensors(device=None, dtype=torch.float32):
    '''
    Precompute action tensors for the current Qa, Qb.
    Call this after configuring P1/P2/P3/P4.
    '''
    global ACTION_CACHE

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    qa_vals = torch.arange(Qa + 1, device=device, dtype=torch.long)
    qb_vals = torch.arange(Qb + 1, device=device, dtype=torch.long)

    qa_grid, qb_grid = torch.meshgrid(qa_vals, qb_vals, indexing="ij")

    qa_flat = qa_grid.reshape(-1)
    qb_flat = qb_grid.reshape(-1)

    order_cost = ca * qa_flat.to(dtype) + cb * qb_flat.to(dtype)

    ACTION_CACHE = {
        "device": device,
        "dtype": dtype,
        "qa_flat": qa_flat,
        "qb_flat": qb_flat,
        "order_cost": order_cost,
        "A": qa_flat.shape[0],
    }

    print("Prepared action tensors:")
    print("device:", device)
    print("A =", qa_flat.shape[0])

### 5.2 Clearing cached probability and event computations

Several functions use `@lru_cache` because probabilities and aggregated events are reused many times.
Whenever `mu_a`, `mu_b`, `Q_a`, `Q_b`, `M`, or `gamma` changes, cached values must be cleared.
This is why `clear_probability_caches()` is called inside the P1–P4 configuration function.


In [14]:
def clear_probability_caches():
    for fn_name in [
        "compute_pu",
        "compute_pa",
        "build_event_for_state",
    ]:
        if fn_name in globals() and hasattr(globals()[fn_name], "cache_clear"):
            globals()[fn_name].cache_clear()

    print("Probability/event caches cleared.")

### 5.3 P1–P4 problem configuration

The paper reports four timing instances P1–P4.
The table's `Qa`/`Qb` values are interpreted as the number of inventory/order levels.
Since the code uses `range(Qa + 1)`, I set the code-level maximum order quantity as:


$Q_{\text{code}} = Q_{\text{levels}} - 1$.


This makes the state counts match the paper exactly.


In [ ]:
def configure_paper_instance(instance_name):
    """
    Configure global variables to match paper P1-P4 timing instances.
    """

    global M, Qa, Qb
    global mu_a, mu_b, sa, sb, ca, cb, gamma
    global states_a, states_b, states_2, state_to_index, N2

    configs = {
        "P1": {"mu_a": 5.0, "mu_b": 5.0, "Qa_levels": 11, "Qb_levels": 11, "paper_N": 14641,},
        "P2": {"mu_a": 5.0, "mu_b": 6.0, "Qa_levels": 11, "Qb_levels": 13, "paper_N": 20449,},
        "P3": {"mu_a": 6.0, "mu_b": 6.0, "Qa_levels": 13, "Qb_levels": 13, "paper_N": 28561,},
        "P4": {"mu_a": 7.0, "mu_b": 7.0, "Qa_levels": 14, "Qb_levels": 14, "paper_N": 38416,},
    }

    if instance_name not in configs:
        raise ValueError("instance_name must be one of: P1, P2, P3, P4")

    cfg = configs[instance_name]

    M = 2
    mu_a = cfg["mu_a"]
    mu_b = cfg["mu_b"]

    sa = 1.0
    sb = 1.0
    ca = 0.5
    cb = 0.5
    gamma = 0.5

    Qa = cfg["Qa_levels"] - 1
    Qb = cfg["Qb_levels"] - 1

    states_a = list(product(range(Qa + 1), repeat=M))
    states_b = list(product(range(Qb + 1), repeat=M))
    states_2 = list(product(states_a, states_b))
    state_to_index = {state: i for i, state in enumerate(states_2)}
    N2 = len(states_2)

    clear_probability_caches()
    prepare_action_tensors()

    print("\n" + "=" * 70)
    print(f"Configured {instance_name}")
    print("=" * 70)
    print("mu_a, mu_b:", mu_a, mu_b)
    print("Qa max, Qb max:", Qa, Qb)
    print("Qa levels, Qb levels:", Qa + 1, Qb + 1)
    print("state count:", N2)
    print("paper N:", cfg["paper_N"])
    print("state count matches paper:", N2 == cfg["paper_N"])
    print("actions per state:", (Qa + 1) * (Qb + 1))
    print("device:", "cuda" if torch.cuda.is_available() else "cpu")

    return cfg

### 5.4 Optional CPU 100-iteration runner

This function can run the full VI loop with the CPU reference implementation.
It is kept for completeness, but it is not used for the final P1–P4 timing because pure Python CPU execution is too slow.


In [16]:
def run_cpu_100_iterations(max_iter = 100, verbose = True):
    N = len(states_2)
    V = np.zeros(N, dtype=np.float64)
    policy = np.zeros((N, 2), dtype = int)

    timing_rows = []
    total_start = time.perf_counter()

    for iteration in range(max_iter):
        iter_start = time.perf_counter()
        W = V.copy()

        for j,state in enumerate(states_2):
            Fv, best_value, best_qa, best_qb = compute_all_values_for_state_cpu(state, W)

            V[j] = best_value
            policy[j, 0] = best_qa
            policy[j, 1] = best_qb

        diff = V - W
        span = diff.max() - diff.min()
        pi_est = 0.5 * (diff.max() + diff.min())

        iter_time = time.perf_counter() - iter_start

        timing_rows.append({
            "method": "CPU_mock",
            "iteration": iteration,
            "iter_time_sec": iter_time,
            "span": span,
            "pi_est": pi_est
        })

        if verbose:
            print(
                f"[CPU] Iteration {iteration:3d}: "
                f"time={iter_time:.3f}s, "
                f"span={span:.8f}, "
                f"pi_est={pi_est:.6f}"
            )

    total_time = time.perf_counter() - total_start

    timing_log = pd.DataFrame(timing_rows)

    print("\n========== CPU Mock 100-Iteration Result ==========")
    print("Total time:", total_time, "sec")
    print("Average iteration time:", timing_log["iter_time_sec"].mean(), "sec")
    print("Final pi_est:", pi_est)
    print("Final span:", span)

    return V, policy, pi_est, timing_log

## 6. Optimized non-batched PyTorch VI

This is the first GPU optimization step.
Compared with the debug version, it:

- keeps \(W\) on GPU for the whole iteration;
- avoids returning the full \(Fv\) matrix to CPU;
- only returns the best action and value for each state.

It is faster than the original state-by-state PyTorch version, but still launches many small computations.


In [ ]:
def torch_compute_best_action_fast(state, W_torch, device=None, dtype=torch.float32):
    """
    Fast VI version for one state.
    Optimizations:
    1. W_torch is passed in and already on GPU.
    2. Does not return full Fv to CPU.
    3. Returns torch scalar tensors, not Python numbers, to avoid per-state CUDA sync.
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    state_a, state_b = state
    Ya = sum(state_a)
    Yb = sum(state_b)

    da_arr, db_arr, prob_arr = build_event_for_state(state)

    da_t = torch.as_tensor(da_arr, device=device, dtype=torch.long)
    db_t = torch.as_tensor(db_arr, device=device, dtype=torch.long)
    prob_t = torch.as_tensor(prob_arr, device=device, dtype=dtype)

    qa_flat = ACTION_CACHE["qa_flat"]
    qb_flat = ACTION_CACHE["qb_flat"]
    order_cost = ACTION_CACHE["order_cost"]
    A = ACTION_CACHE["A"]

    E = da_t.shape[0]

    qa_mat = qa_flat[:, None].expand(A, E)
    qb_mat = qb_flat[:, None].expand(A, E)

    da_mat = da_t[None, :].expand(A, E)
    db_mat = db_t[None, :].expand(A, E)

    next_state_a = next_inventory_torch(state_a, qa_mat, da_mat, Qa, M)
    next_state_b = next_inventory_torch(state_b, qb_mat, db_mat, Qb, M)

    index_a = encode_inventory_torch(next_state_a, Qa, M)
    index_b = encode_inventory_torch(next_state_b, Qb, M)

    num_states_b = (Qb + 1) ** M
    next_index = (index_a * num_states_b + index_b).long()

    Ya_t = torch.tensor(Ya, device=device, dtype=torch.long)
    Yb_t = torch.tensor(Yb, device=device, dtype=torch.long)

    sale_a = torch.minimum(da_mat, Ya_t)
    sale_b = torch.minimum(db_mat, Yb_t)

    reward = sa * sale_a.to(dtype) + sb * sale_b.to(dtype)
    future = W_torch[next_index]
    values = (prob_t[None, :] * (reward + future)).sum(dim=1)
    values = values - order_cost

    best_idx = torch.argmax(values)

    best_value = values[best_idx]
    best_qa = qa_flat[best_idx]
    best_qb = qb_flat[best_idx]

    return best_value, best_qa, best_qb

In [ ]:
def run_torch_100_iterations(max_iter=100, verbose=True, print_every=1, dtype=torch.float32, stop_on_convergence=False, epsilon=1e-4):
    """
    Optimized Torch VI runner.
    Key changes:
    1. V is kept on GPU during iterations.
    2. W_torch = V_torch.clone() once per iteration.
    3. Uses torch_compute_best_action_fast.
    4. Copies V/policy back to CPU only at the end.
    """

    N = len(states_2)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)
    if device == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    prepare_action_tensors(device=device, dtype=dtype)

    # Keep V and policy on GPU
    V_torch = torch.zeros(N, device=device, dtype=dtype)
    policy_torch = torch.zeros((N, 2), device=device, dtype=torch.long)

    timing_rows = []

    if device == "cuda":
        torch.cuda.synchronize()

    total_start = time.perf_counter()

    with torch.no_grad():
        for iteration in range(max_iter):
            W_torch = V_torch.clone()

            V_new = torch.empty_like(V_torch)
            policy_new = torch.empty((N, 2), device=device, dtype=torch.long)

            if device == "cuda":
                torch.cuda.synchronize()

            iter_start = time.perf_counter()

            for j, state in enumerate(states_2):
                best_value, best_qa, best_qb = torch_compute_best_action_fast(
                    state=state,
                    W_torch=W_torch,
                    device=device,
                    dtype=dtype,
                )

                V_new[j] = best_value
                policy_new[j, 0] = best_qa
                policy_new[j, 1] = best_qb

            # Synchronize once per full iteration, not once per state
            if device == "cuda":
                torch.cuda.synchronize()

            iter_time = time.perf_counter() - iter_start

            diff = V_new - W_torch
            span_t = diff.max() - diff.min()
            pi_t = 0.5 * (diff.max() + diff.min())

            # Only two scalar synchronizations per iteration
            span = float(span_t.item())
            pi_est = float(pi_t.item())

            V_torch = V_new
            policy_torch = policy_new

            timing_rows.append({
                "method": "Torch_fast",
                "iteration": iteration,
                "iter_time_sec": iter_time,
                "span": span,
                "pi_est": pi_est,
            })

            if verbose and (iteration % print_every == 0 or iteration == max_iter - 1):
                print(
                    f"[Torch-fast] Iteration {iteration:3d}: "
                    f"time={iter_time:.3f}s, "
                    f"span={span:.8f}, "
                    f"pi_est={pi_est:.6f}"
                )

            if stop_on_convergence and span < epsilon:
                print(f"Converged at iteration {iteration}; stopping early.")
                break

    if device == "cuda":
        torch.cuda.synchronize()

    total_time = time.perf_counter() - total_start

    # Copy back only once at the end
    V = V_torch.detach().cpu().numpy()
    policy = policy_torch.detach().cpu().numpy()

    timing_log = pd.DataFrame(timing_rows)

    print("\n========== Torch-fast 100-Iteration Result ==========")
    print("Total wall time:", total_time, "sec")
    print("Sum of iteration times:", timing_log["iter_time_sec"].sum(), "sec")
    print("Average iteration time:", timing_log["iter_time_sec"].mean(), "sec")
    print("Final pi_est:", pi_est)
    print("Final span:", span)

    return V, policy, pi_est, timing_log

## 7. Paper Table 4 reference values

This DataFrame records the paper's reported P1–P4 runtimes.
It is used to compare my PyTorch GPU implementation against the paper's sequential and CUDA GPU runtimes.


In [ ]:
paper_timing_reference = pd.DataFrame({
    "instance": ["P1", "P2", "P3", "P4"],
    "paper_N": [14641, 20449, 28561, 38416],
    "paper_seq_total_sec": [357.89, 698.20, 1358.04, 4241.66],
    "paper_gpu_total_sec": [91.29, 136.84, 214.71, 361.49],
    "paper_seq_substitution_sec": [351.31, 688.71, 1344.82, 4217.15],
    "paper_gpu_substitution_sec": [39.94, 67.29, 111.99, 215.54],
})

paper_timing_reference["paper_total_speedup"] = (
    paper_timing_reference["paper_seq_total_sec"]
    / paper_timing_reference["paper_gpu_total_sec"]
)

paper_timing_reference["paper_substitution_speedup"] = (
    paper_timing_reference["paper_seq_substitution_sec"]
    / paper_timing_reference["paper_gpu_substitution_sec"]
)

paper_timing_reference

### 7.1 Generic 100-iteration comparison runner

This function wraps the CPU and PyTorch VI runners and produces a summary table.
It is useful for testing different implementations under a unified interface.


In [20]:
# compare cpu vs torch runtime over 100 iterations
def print_current_config():
    print("Current configuration:")
    print("M =", M)
    print("Qa =", Qa)
    print("Qb =", Qb)
    print("mu_a =", mu_a)
    print("mu_b =", mu_b)
    print("Number of states =", len(states_2))
    print("Number of actions per state =", (Qa + 1) * (Qb + 1))
    print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


def run_100_iteration_comparison(instance_name = None, max_iter = 100, run_cpu = True, run_gpu = True, verbose = True):
    if instance_name is not None:
        configure_paper_instance(instance_name)
    print_current_config()
    results = {}

    if run_cpu:
        print("\n" + "=" * 80)
        print("Running full CPU mock 100 iterations")
        print("=" * 80)

        V_cpu, policy_cpu, pi_cpu, cpu_log = run_cpu_100_iterations(
            max_iter=100,
            verbose=True
        )

        results["cpu"] = {
            "V": V_cpu,
            "policy": policy_cpu,
            "pi": pi_cpu,
            "log": cpu_log,
            "total_time": cpu_log["iter_time_sec"].sum(),
            "avg_iter_time": cpu_log["iter_time_sec"].mean(),
            "final_span": cpu_log["span"].iloc[-1],
        }

    if run_gpu:
        print("\n" + "=" * 80)
        print("Running full Torch 100 iterations")
        print("=" * 80)

        V_torch, policy_torch, pi_torch, torch_log = run_torch_100_iterations(
            max_iter=100,
            verbose=True
        )

        results["torch"] = {
            "V": V_torch,
            "policy": policy_torch,
            "pi": pi_torch,
            "log": torch_log,
            "total_time": torch_log["iter_time_sec"].sum(),
            "avg_iter_time": torch_log["iter_time_sec"].mean(),
            "final_span": torch_log["span"].iloc[-1],
        }

    rows = []

    if "cpu" in results:
        rows.append({
            "instance": instance_name,
            "method": "CPU_mock",
            "iterations": len(results['cpu']['log']),
            "N": len(states_2),
            "Qa_max": Qa,
            "Qb_max": Qb,
            "Qa_levels": Qa + 1,
            "Qb_levels": Qb + 1,
            "mu_a": mu_a,
            "mu_b": mu_b,
            "total_iter_sec": results["cpu"]["total_time"],
            "avg_iter_sec": results["cpu"]["avg_iter_time"],
            "final_pi_est": results["cpu"]["pi"],
            "final_span": results["cpu"]["final_span"],
        })

    if "torch" in results:
        rows.append({
            "instance": instance_name,
            "method": "Torch",
            "iterations": len(results["torch"]['log']),
            "N": len(states_2),
            "Qa_max": Qa,
            "Qb_max": Qb,
            "Qa_levels": Qa + 1,
            "Qb_levels": Qb + 1,
            "mu_a": mu_a,
            "mu_b": mu_b,
            "total_iter_sec": results["torch"]["total_time"],
            "avg_iter_sec": results["torch"]["avg_iter_time"],
            "final_pi_est": results["torch"]["pi"],
            "final_span": results["torch"]["final_span"],
        })


    summary_df = pd.DataFrame(rows)

    if "cpu" in results and "torch" in results:
        cpu_time = results["cpu"]["total_time"]
        torch_time = results["torch"]["total_time"]
        speedup = cpu_time / torch_time

        print("\n========== Timing Summary ==========")
        print(summary_df)
        print()
        print("Speedup CPU / Torch:", speedup)

        V_diff = np.max(np.abs(results["cpu"]["V"] - results["torch"]["V"]))
        policy_same = np.array_equal(results["cpu"]["policy"], results["torch"]["policy"])

        print("Max final V abs diff:", V_diff)
        print("Final policy exactly same:", policy_same)

    else:
        print("\n========== Timing Summary ==========")
        print(summary_df)

    if instance_name in ["P1", "P2", "P3", "P4"]:
        paper_row = paper_timing_reference[paper_timing_reference["instance"] == instance_name]

        comparison_df = summary_df.merge(
            paper_row,
            on="instance",
            how="left"
        )

        if "torch" in results:
            comparison_df["your_torch_vs_paper_gpu_ratio"] = (
                comparison_df["total_iter_sec"] / comparison_df["paper_gpu_total_sec"]
            )

        print("\n========== Comparison with Paper Table 4 ==========")
        display_cols = [
            "instance",
            "method",
            "N",
            "paper_N",
            "iterations",
            "total_iter_sec",
            "avg_iter_sec",
            "paper_gpu_total_sec",
            "paper_seq_total_sec",
            "paper_total_speedup",
            "paper_substitution_speedup",
            "final_pi_est",
            "final_span",
        ]

        existing_cols = [c for c in display_cols if c in comparison_df.columns]
        print(comparison_df[existing_cols])

    else:
        comparison_df = summary_df.copy()

    return results, summary_df, comparison_df

### 7.2 Running P1–P4 with the comparison wrapper

This wrapper repeatedly configures P1–P4 and runs the selected VI implementation.
It was useful during development; the final result below uses the batched VI runner directly.


In [21]:
def run_p1_to_p4_with_existing_comparison(max_iter=100, run_cpu=False, run_gpu=True, verbose=True):
    """
    Run P1-P4 by repeatedly calling the modified run_100_iteration_comparison.
    """
    all_results = {}
    summary_list = []
    comparison_list = []

    for inst in ["P1", "P2", "P3", "P4"]:
        print("\n" + "#" * 100)
        print(f"Running {inst}")
        print("#" * 100)

        results, summary_df, comparison_df = run_100_iteration_comparison(inst, max_iter, run_cpu, run_gpu, verbose)

        all_results[inst] = results
        summary_list.append(summary_df)
        comparison_list.append(comparison_df)

    all_summary = pd.concat(summary_list, ignore_index=True)
    all_comparison = pd.concat(comparison_list, ignore_index=True)

    print("\n" + "=" * 100)
    print("All P1-P4 Summary")
    print("=" * 100)
    print(all_comparison)

    return all_results, all_summary, all_comparison

## 8. Final batched PyTorch implementation

The main bottleneck of the non-batched version is the Python loop over states.
The batched version processes multiple states at once by padding each state's aggregated event list to a common event length $E_{\max}$.

Conceptually, the computation becomes an action-state-event tensor reduction:

$
\text{values}[b,a] =
\sum_e p[b,e]\left(r[b,a,e] + W[\text{next}[b,a,e]]\right).
$

This better matches the parallel structure of the paper's CUDA implementation.


In [22]:
## batch version
def prepare_state_event_tensors(device=None, dtype=torch.float32):
    """
    Precompute:
    - state_a_all: (N, M)
    - state_b_all: (N, M)
    - da_pad:      (N, Emax)
    - db_pad:      (N, Emax)
    - prob_pad:    (N, Emax)

    This avoids calling build_event_for_state and tensor conversion inside every iteration/state.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    N = len(states_2)

    state_a_np = np.array([s[0] for s in states_2], dtype=np.int64)
    state_b_np = np.array([s[1] for s in states_2], dtype=np.int64)

    event_data = []
    Emax = 0

    for state in states_2:
        da_arr, db_arr, prob_arr = build_event_for_state(state)
        event_data.append((da_arr, db_arr, prob_arr))
        Emax = max(Emax, len(prob_arr))

    da_pad = np.zeros((N, Emax), dtype=np.int64)
    db_pad = np.zeros((N, Emax), dtype=np.int64)
    prob_pad = np.zeros((N, Emax), dtype=np.float32)

    for i, (da_arr, db_arr, prob_arr) in enumerate(event_data):
        E = len(prob_arr)
        da_pad[i, :E] = da_arr
        db_pad[i, :E] = db_arr
        prob_pad[i, :E] = prob_arr

    state_a_all = torch.as_tensor(state_a_np, device=device, dtype=torch.long)
    state_b_all = torch.as_tensor(state_b_np, device=device, dtype=torch.long)

    da_all = torch.as_tensor(da_pad, device=device, dtype=torch.long)
    db_all = torch.as_tensor(db_pad, device=device, dtype=torch.long)
    prob_all = torch.as_tensor(prob_pad, device=device, dtype=dtype)

    event_cache = {
        "device": device,
        "dtype": dtype,
        "state_a_all": state_a_all,
        "state_b_all": state_b_all,
        "da_all": da_all,
        "db_all": db_all,
        "prob_all": prob_all,
        "Emax": Emax,
        "N": N,
    }

    print("Prepared state-event tensors")
    print("N =", N)
    print("Emax =", Emax)
    print("device =", device)

    return event_cache

### 8.1 Batched FIFO transition

This function generalizes the FIFO transition to a batch of states.
Its output components have shape `(B, A, E)`, where:

- `B` = batch size;
- `A` = number of action pairs;
- `E` = padded number of aggregated events.


In [23]:
def next_inventory_batch_torch(state_batch, q_flat, demand_batch, Q, M):
    """
    General-M FIFO transition for batched states.

    state_batch:   (B, M)
    q_flat:        (A,)
    demand_batch:  (B, E)

    Return:
        list length M, each tensor shape (B, A, E)
    """
    B = state_batch.shape[0]
    A = q_flat.shape[0]
    E = demand_batch.shape[1]
    device = state_batch.device

    # demand: (B, 1, E)
    d = demand_batch[:, None, :]

    # q: (1, A, 1)
    q = q_flat[None, :, None].expand(B, A, E)

    next_components = []

    prefix = torch.zeros(B, device=device, dtype=torch.long)

    for r in range(M - 1):
        prefix = prefix + state_batch[:, r]

        unmet = torch.clamp(d - prefix[:, None, None], min=0)

        next_r = torch.clamp(
            state_batch[:, r + 1][:, None, None] - unmet,
            min=0,
            max=Q
        )

        next_components.append(next_r.long())

    next_components.append(q.long())

    return next_components

In [24]:
def encode_inventory_batch_torch(inv_components, Q, M):
    """
    inv_components: list length M, each tensor shape (B, A, E)

    Return:
        index tensor shape (B, A, E)
    """
    idx = torch.zeros_like(inv_components[0], dtype=torch.long)
    base = Q + 1

    for r in range(M):
        power = M - 1 - r
        idx = idx + inv_components[r].long() * (base ** power)

    return idx

### 8.2 Batched best-action computation

For each batch of states, this function computes all action values in parallel and then takes `argmax`
over the action dimension. This is the core GPU computation used in the final reproduction.


In [ ]:
def torch_compute_batch_best_actions_fast(batch_idx, W_torch, event_cache, device=None, dtype=torch.float32):
    """
    Batch version of torch_compute_best_action_fast.

    batch_idx:
        1D tensor/list/array of state indices, length B.

    Return:
        best_values: (B,)
        best_qas:    (B,)
        best_qbs:    (B,)
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if not torch.is_tensor(batch_idx):
        batch_idx = torch.as_tensor(batch_idx, device=device, dtype=torch.long)
    else:
        batch_idx = batch_idx.to(device=device, dtype=torch.long)

    state_a = event_cache["state_a_all"][batch_idx]   # (B, M)
    state_b = event_cache["state_b_all"][batch_idx]   # (B, M)

    da = event_cache["da_all"][batch_idx]             # (B, E)
    db = event_cache["db_all"][batch_idx]             # (B, E)
    prob = event_cache["prob_all"][batch_idx]         # (B, E)

    B = state_a.shape[0]
    E = da.shape[1]

    qa_flat = ACTION_CACHE["qa_flat"]                 # (A,)
    qb_flat = ACTION_CACHE["qb_flat"]                 # (A,)
    order_cost = ACTION_CACHE["order_cost"]           # (A,)
    A = ACTION_CACHE["A"]

    # Next states: each component shape (B, A, E)
    next_a_components = next_inventory_batch_torch(state_batch=state_a, q_flat=qa_flat, demand_batch=da, Q=Qa, M=M)
    next_b_components = next_inventory_batch_torch(state_batch=state_b, q_flat=qb_flat, demand_batch=db, Q=Qb, M=M)

    idx_a = encode_inventory_batch_torch(next_a_components, Qa, M)
    idx_b = encode_inventory_batch_torch(next_b_components, Qb, M)

    num_states_b = (Qb + 1) ** M
    next_idx = (idx_a * num_states_b + idx_b).long()  # (B, A, E)

    Ya = state_a.sum(dim=1)                           # (B,)
    Yb = state_b.sum(dim=1)                           # (B,)

    sale_a = torch.minimum(
        da[:, None, :].expand(B, A, E),
        Ya[:, None, None]
    )

    sale_b = torch.minimum(
        db[:, None, :].expand(B, A, E),
        Yb[:, None, None]
    )

    reward = sa * sale_a.to(dtype) + sb * sale_b.to(dtype)   # (B, A, E)

    future = W_torch[next_idx]                               # (B, A, E)

    values = (
        prob[:, None, :] * (reward + future)
    ).sum(dim=2)                                             # (B, A)

    values = values - order_cost[None, :]

    best_idx = torch.argmax(values, dim=1)                   # (B,)

    batch_arange = torch.arange(B, device=device)

    best_values = values[batch_arange, best_idx]
    best_qas = qa_flat[best_idx]
    best_qbs = qb_flat[best_idx]

    return best_values, best_qas, best_qbs

### 8.3 Batched value-iteration runner

This is the final VI implementation used for the Table 4 reproduction.
It keeps value vectors and policies on GPU during iterations and only transfers final results back to CPU.

The final selected setting is:

$
\texttt{dtype=torch.float64}, \qquad \texttt{batch\_size=256}.
$

This gives stable convergence while maintaining runtime close to the paper's GPU results.


In [ ]:
def run_torch_100_iterations_batched(max_iter=100, batch_size=64, verbose=True, print_every=1, dtype=torch.float32, stop_on_convergence=False, epsilon=5e-4):
    """
    Batched PyTorch VI.
    Faster than per-state torch_compute_best_action_fast.
    """
    N = len(states_2)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)
    if device == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))

    prepare_action_tensors(device=device, dtype=dtype)

    event_cache = prepare_state_event_tensors(device=device, dtype=dtype)

    V_torch = torch.zeros(N, device=device, dtype=dtype)
    policy_torch = torch.zeros((N, 2), device=device, dtype=torch.long)

    timing_rows = []

    if device == "cuda":
        torch.cuda.synchronize()

    total_start = time.perf_counter()

    with torch.no_grad():
        for iteration in range(max_iter):
            W_torch = V_torch.clone()

            V_new = torch.empty_like(V_torch)
            policy_new = torch.empty_like(policy_torch)

            if device == "cuda":
                torch.cuda.synchronize()

            iter_start = time.perf_counter()

            for start in range(0, N, batch_size):
                end = min(start + batch_size, N)

                batch_idx = torch.arange(start, end, device=device, dtype=torch.long)

                best_values, best_qas, best_qbs = torch_compute_batch_best_actions_fast(
                    batch_idx=batch_idx,
                    W_torch=W_torch,
                    event_cache=event_cache,
                    device=device,
                    dtype=dtype,
                )

                V_new[batch_idx] = best_values
                policy_new[batch_idx, 0] = best_qas
                policy_new[batch_idx, 1] = best_qbs

            if device == "cuda":
                torch.cuda.synchronize()

            iter_time = time.perf_counter() - iter_start

            diff = V_new - W_torch
            span_t = diff.max() - diff.min()
            pi_t = 0.5 * (diff.max() + diff.min())

            span = float(span_t.item())
            pi_est = float(pi_t.item())

            V_torch = V_new
            policy_torch = policy_new

            timing_rows.append({
                "method": "Torch_batched",
                "iteration": iteration,
                "iter_time_sec": iter_time,
                "span": span,
                "pi_est": pi_est,
                "batch_size": batch_size,
            })

            if verbose and (iteration % print_every == 0 or iteration == max_iter - 1):
                print(
                    f"[Torch-batched] Iteration {iteration:3d}: "
                    f"time={iter_time:.3f}s, "
                    f"span={span:.8f}, "
                    f"pi_est={pi_est:.6f}"
                )

            if stop_on_convergence and span < epsilon:
                print(f"Converged at iteration {iteration}; stopping early.")
                break

    if device == "cuda":
        torch.cuda.synchronize()

    total_time = time.perf_counter() - total_start

    V = V_torch.detach().cpu().numpy()
    policy = policy_torch.detach().cpu().numpy()

    timing_log = pd.DataFrame(timing_rows)

    print("\n========== Torch-batched Result ==========")
    print("Total wall time:", total_time, "sec")
    print("Sum of iteration times:", timing_log["iter_time_sec"].sum(), "sec")
    print("Average iteration time:", timing_log["iter_time_sec"].mean(), "sec")
    print("Final pi_est:", pi_est)
    print("Final span:", span)

    return V, policy, pi_est, timing_log

## 9. Final reproduction run: P1–P4 Table 4

This cell runs the final batched PyTorch implementation on P1–P4 for 100 iterations.
It uses the final selected configuration:

- `dtype = torch.float64`
- `batch_size = 256`
- `max_iter = 100`

The result table reports runtime, estimated average profit, final span, and runtime ratio relative to the paper's GPU time.


In [ ]:
# Final reproduction run: Table 4 P1-P4 timing
# Recommended configuration based on speed/accuracy tradeoff:
#   dtype = torch.float64
#   batch_size = 256
#   max_iter = 100

final_results = {}

for inst in ["P1", "P2", "P3", "P4"]:
    configure_paper_instance(inst)
    V_b, policy_b, pi_b, log_b = run_torch_100_iterations_batched(
        max_iter=100,
        batch_size=256,
        verbose=True,
        print_every=5,
        dtype=torch.float64,
        stop_on_convergence=False,
    )

    final_results[inst] = {
        "V": V_b,
        "policy": policy_b,
        "pi_est": pi_b,
        "log": log_b,
        "total_time_sec": log_b["iter_time_sec"].sum(),
        "avg_iter_time_sec": log_b["iter_time_sec"].mean(),
        "final_span": log_b["span"].iloc[-1],
    }

# Build a compact summary table.
final_summary = pd.DataFrame([
    {
        "instance": inst,
        "total_time_sec": res["total_time_sec"],
        "avg_iter_time_sec": res["avg_iter_time_sec"],
        "final_pi_est": res["pi_est"],
        "final_span": res["final_span"],
    }
    for inst, res in final_results.items()
])

final_summary = final_summary.merge(
    paper_timing_reference[["instance", "paper_gpu_total_sec", "paper_seq_total_sec"]],
    on="instance",
    how="left",
)

final_summary["runtime_ratio_vs_paper_gpu"] = (
    final_summary["total_time_sec"] / final_summary["paper_gpu_total_sec"]
)

final_summary


## 10. Final result summary from the selected configuration

The following results were obtained using `torch.float64` and `batch_size=256` on a Tesla T4 GPU.

| Instance | Paper GPU time (s) | Batched PyTorch time (s) | Runtime ratio | Final `pi_est` | Final span |
|---|---:|---:|---:|---:|---:|
| P1 | 91.29 | 89.41 | 0.98× | 4.503203516 | 2.84e-7 |
| P2 | 136.84 | 173.74 | 1.27× | 5.003599909 | 3.17e-7 |
| P3 | 214.71 | 340.33 | 1.59× | 5.508719865 | 3.49e-7 |
| P4 | 361.49 | 616.00 | 1.70× | 6.518342721 | 3.96e-7 |

Overall, the reproduction matches the paper's P1–P4 state sizes, follows the same 100-iteration timing protocol,
and produces converged VI results with final spans below $4\times10^{-7}$.
The remaining runtime gap for P2–P4 is expected because this notebook uses batched PyTorch operations rather than hand-written CUDA kernels.
